<a href="https://colab.research.google.com/github/technerd6996/AI-Learning/blob/main/Triage_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **SRE Triage Agent**

---



---




In [1]:
from google.colab import drive #cell 1
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip install -q autoawq gptqmodel streamlit pyngrok #cell 2
!pip install -q protobuf==7.34.0

In [3]:
from google.colab import drive # cell 3 -Restart after executing Cell 2 and Run Cell 3
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os #CELL 4
from transformers import AutoModelForCausalLM, AutoTokenizer

DRIVE_CACHE_DIR = "/content/drive/MyDrive/triage-agent/hf_cache"
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct-AWQ"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=DRIVE_CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=DRIVE_CACHE_DIR,
    device_map="auto",
)
print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.5.0
Transformers : 5.16.1
Torch        : 2.11.0+cu128
Triton       : 3.6.0
INFO  ExLlamaV2 AWQ: compiling torch.ops JIT extension in `/root/.cache/gptqmodel/torch_

Loading weights:   0%|          | 0/1251 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

INFO  gc.collect() reclaimed 11101 objects in 0.351s                            
Model loaded on: cuda:0


In [6]:
del model, tokenizer #NOT A CELL RUN IT AFTER IMPORITING THE MODEL
import torch, gc
torch.cuda.empty_cache()
gc.collect()

NameError: name 'model' is not defined

In [4]:
%%writefile system_prompt.py
# Cell 5
SYSTEM_PROMPT = """You are TRIAGE-1, a Level 1 Site Reliability Engineer (SRE) on call.

Your job: a developer will paste an error snippet, stack trace, or log excerpt.
You must analyze it and respond in EXACTLY the format below. Do not add
greetings, disclaimers, or text outside this structure.

OUTPUT FORMAT (always use these exact section headers):

SEVERITY: <INFO | WARN | CRITICAL>

ROOT CAUSE HYPOTHESIS:
<1-3 sentences. State your best hypothesis. If the log is ambiguous, say so
explicitly rather than guessing with false confidence.>

RESOLUTION CHECKLIST:
- [ ] <first concrete diagnostic or fix step>
- [ ] <second step>
- [ ] <third step, add more if genuinely needed, don't pad>

RULES FOR CLASSIFICATION:
- INFO: expected/benign log lines, informational noise, successful operations
  logged at a high verbosity level. No action needed.
- WARN: something degraded, retried, or approaching a threshold, but the
  system is still functioning (e.g. connection pool nearing capacity, a
  retried request that eventually succeeded, deprecated API usage).
- CRITICAL: active failure, data loss risk, service unavailability, security
  exposure, or unhandled exceptions crashing a process.

RULES FOR YOUR BEHAVIOR:
- Never fabricate a root cause you're not reasonably confident in — say
  "insufficient information to determine root cause" and list what
  additional logs/context you'd need, instead of inventing a plausible-
  sounding guess.
- Never skip a section, never rename a section header, never add extra
  sections.
- If given input that is not an error log or stack trace at all, still use
  the format, but state that clearly under ROOT CAUSE HYPOTHESIS and
  mark SEVERITY as INFO.
"""

Overwriting system_prompt.py


In [5]:
%%writefile .streamlit/config.toml
[theme]
base = "dark"
primaryColor = "#ffb000"
backgroundColor = "#0a0d10"
secondaryBackgroundColor = "#12161c"
textColor = "#d5dbe3"
font = "monospace"

Overwriting .streamlit/config.toml


In [10]:
%%writefile app.py
import os
import re
import shutil
import streamlit as st
from transformers import AutoModelForCausalLM, AutoTokenizer
from system_prompt import SYSTEM_PROMPT

DRIVE_CACHE_DIR = "/content/drive/MyDrive/triage-agent/hf_cache"
LOCAL_CACHE_DIR = "/content/local_hf_cache"
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct-AWQ"

st.set_page_config(page_title="SRE L1 TRIAGE AGENT", page_icon="▮", layout="centered")

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;500;700&display=swap');
html, body, [class*="css"], .stApp, .stMarkdown, .stChatMessage, .stTextInput, .stButton {
    font-family: 'JetBrains Mono', monospace !important;
}
.stApp { background-color: #0a0d10; }
section[data-testid="stSidebar"] { background-color: #0d1116; border-right: 1px solid #1e242c; }
.stChatMessage {
    background-color: transparent !important;
    border-bottom: 1px solid #161b21;
    border-radius: 0 !important;
    padding: 14px 0 !important;
}
.console-header {
    display: flex; align-items: center; gap: 10px;
    padding: 6px 0 18px 0; border-bottom: 1px solid #1e242c; margin-bottom: 22px;
    color: #707b87; font-size: 0.8rem; letter-spacing: 0.5px;
}
.console-dot {
    width: 8px; height: 8px; border-radius: 50%; background: #ffb000;
    box-shadow: 0 0 6px #ffb000; animation: pulse 2s ease-in-out infinite;
}
@keyframes pulse { 0%,100% { opacity: 1; } 50% { opacity: 0.35; } }
h1 { font-weight: 700 !important; letter-spacing: -0.5px; color: #e8ecf1 !important; }
.severity-line {
    font-size: 0.95rem; font-weight: 700; letter-spacing: 1px; margin-bottom: 10px;
    padding-left: 12px; border-left: 3px solid;
}
.sev-INFO     { color: #6fd3ff; border-color: #6fd3ff; }
.sev-WARN     { color: #ffb000; border-color: #ffb000; }
.sev-CRITICAL { color: #ff5252; border-color: #ff5252; }
.file-chip {
    display: inline-flex; align-items: center; gap: 8px;
    border: 1px solid #2a323c; background: #12161c; color: #9aa4b0;
    padding: 5px 12px; font-size: 0.82rem;
}
.stButton button {
    background-color: #12161c !important; color: #d5dbe3 !important;
    border: 1px solid #2a323c !important; border-radius: 2px !important;
    font-family: 'JetBrains Mono', monospace !important;
}
.stButton button:hover { border-color: #ffb000 !important; color: #ffb000 !important; }
</style>
""", unsafe_allow_html=True)

@st.cache_resource(show_spinner=False)
def load_model():
    with st.status("initializing triage-agent-1...", expanded=True) as status:
        if not os.path.exists(LOCAL_CACHE_DIR):
            status.update(label="syncing model weights to local disk...")
            shutil.copytree(DRIVE_CACHE_DIR, LOCAL_CACHE_DIR)
        status.update(label="loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=LOCAL_CACHE_DIR)
        status.update(label="loading model weights onto gpu...")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, cache_dir=LOCAL_CACHE_DIR, device_map="auto", low_cpu_mem_usage=True,
        )
        status.update(label="triage-agent-1 online", state="complete", expanded=False)
    return tokenizer, model

tokenizer, model = load_model()

if "history" not in st.session_state:
    st.session_state.history = [{"role": "system", "content": SYSTEM_PROMPT}]
if "uploader_key" not in st.session_state:
    st.session_state.uploader_key = 0

with st.sidebar:
    st.markdown("### ▮ L1 TRIAGE AGENT")
    st.caption("L1 SRE Log Analyser")
    st.markdown("---")
    st.markdown(f"`model` :: {MODEL_ID.split('/')[-1]}")
    st.markdown(f"`turns`  :: {max(0, len(st.session_state.history) - 1) // 2}")
    st.markdown("---")
    if st.button("clear session"):
        st.session_state.history = [{"role": "system", "content": SYSTEM_PROMPT}]
        st.session_state.uploader_key += 1   # forces a fresh, empty file uploader
        st.rerun()

st.markdown(
    '<div class="console-header"><span class="console-dot"></span> triage-1 // listening</div>',
    unsafe_allow_html=True,
)
st.title("L1 TRIAGE AGENT")
st.caption("paste an error, or attach a log file, below")

def render_response(content):
    match = re.search(r"SEVERITY:\s*(INFO|WARN|CRITICAL)", content)
    if match:
        sev = match.group(1)
        st.markdown(f'<div class="severity-line sev-{sev}">SEVERITY :: {sev}</div>', unsafe_allow_html=True)
        content = content[match.end():].lstrip()
    st.markdown(content)

def run_triage(model_content, display_content=None):
    st.session_state.history.append({
        "role": "user", "content": model_content, "display": display_content or model_content,
    })
    prompt = tokenizer.apply_chat_template(
        [{"role": m["role"], "content": m["content"]} for m in st.session_state.history],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with st.spinner("analyzing log..."):
        output = model.generate(**inputs, max_new_tokens=400, temperature=0.3, do_sample=True)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    st.session_state.history.append({"role": "assistant", "content": response, "display": response})

for msg in st.session_state.history[1:]:
    with st.chat_message(msg["role"]):
        if msg["role"] == "assistant":
            render_response(msg["display"])
        else:
            st.markdown(msg["display"], unsafe_allow_html=True)

uploaded_file = st.file_uploader(
    "attach a log file (optional)", type=["txt", "log", "json"],
    key=f"uploader_{st.session_state.uploader_key}",
)
if uploaded_file is not None:
    file_text = uploaded_file.read().decode("utf-8", errors="ignore")
    uploaded_file.seek(0)
    with st.expander(f"preview :: {uploaded_file.name}", expanded=False):
        st.code(file_text[:2000], language=None)
    if st.button("▸ analyze file", type="primary"):
        run_triage(
            model_content=f"[Attached file: {uploaded_file.name}]\n{file_text}",
            display_content=f'📄 {uploaded_file.name}',
        )
        st.session_state.uploader_key += 1   # clears the uploader after use
        st.rerun()

user_typed = st.chat_input("paste an error snippet or stack trace...")
if user_typed:
    run_triage(user_typed)
    st.rerun()

Overwriting app.py


In [7]:
#cell 7 Ngrok AUth

from pyngrok import ngrok
ngrok.set_auth_token("3JvJLNfRUvkfeWP7sQo5pfGFn1Z_3b8poHVgVyEEpgpX9RhVb")

In [8]:
#Cell 8

!pkill -f streamlit
!streamlit run app.py &>/content/logs.txt &

import time
time.sleep(5)
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://cushy-excavator-prism.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!pkill -9 -f streamlit
!pkill -9 -f python